# Resnet101 - MNIST - Basic model

# Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
import os
import sys
sys.path.insert(0,"..")
from utils import *



from torchvision.models import resnet101

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' The Device is set to : {device}')

# Import dataset

- augmented
- normalized
- padded
- shuffled
- Mnist

In [2]:
trainloader, testloader, trainset, testset = load_mnist(BATCH_SIZE=32,PATH="./data")

100%|██████████| 9912422/9912422 [00:08<00:00, 1148137.44it/s]


Extracting ./data\MNIST\raw\train-images-idx3-ubyte.gz to ./data\MNIST\raw



100%|██████████| 28881/28881 [00:00<00:00, 263254.91it/s]


Extracting ./data\MNIST\raw\train-labels-idx1-ubyte.gz to ./data\MNIST\raw



100%|██████████| 1648877/1648877 [00:01<00:00, 1051008.35it/s]


Extracting ./data\MNIST\raw\t10k-images-idx3-ubyte.gz to ./data\MNIST\raw



100%|██████████| 4542/4542 [00:00<?, ?it/s]

Extracting ./data\MNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\MNIST\raw



# Model : Resnet101

In [3]:
model = resnet101()
model.fc = nn.Linear(in_features= 2048, out_features=10, bias= True)
# model.avgpool = nn.Identity()
# model.avgpool = nn.AdaptiveAvgPool2d((output_size=(6, 6)))
model = model.to(device)

num_params = count_param(model)

print("number of parameters:" , num_params)
print(model)


create model
number of parameters: 7978856


# Model Train and Evaluation

In [4]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr= 0.1,
                        momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=200)


In [ ]:
def topk(output, target, k):
    correct = 0.0
    batch_size = output.shape[0]
    for sample in range(batch_size):
        topk_sorted = output[sample].sort()[1][:k]
        if target[sample] in topk_sorted:
           correct+=1
        #    print(f'sample {sample} was correct because : {topk_sorted} and {target[sample]}')
    return (correct/batch_size)*100.0
        

In [6]:

def train(epoch):
    file_path = '../results/mnist/basic_model/resnet101_train.txt'
    
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    print('\nEpoch: %d' % epoch)
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    with open(file_path, 'a') as f:
        f.write(f'\nEpoch: {epoch}\n')
        
        for batch_idx, (inputs, targets) in enumerate(trainloader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        train_summary = f'Train Summary after Epoch: {epoch}, Loss: {train_loss / len(trainloader):.3f}, Accuracy: {100. * correct / total:.3f}% ({correct}/{total})\n'
        f.write(train_summary)

        print(train_summary)

        model_save_path = f'../results/mnist/basic_model/epoch_{epoch}_Resnet101_train.pth'
        os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
        torch.save(model.state_dict(), model_save_path)
        print(f'Model saved to {model_save_path}')



In [7]:
def test(epoch):
    file_path = '../results/mnist/basic_model/resnet101_test.txt'
    
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        with open(file_path, 'a') as f:
            f.write(f'\nTesting after Epoch: {epoch}\n')
            
            for batch_idx, (inputs, targets) in enumerate(testloader):
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
                
            test_summary = f'Test Summary after Epoch {epoch}, Loss: {test_loss / len(testloader):.3f}, Accuracy: {100. * correct / total:.3f}% ({correct}/{total})\n'
            f.write(test_summary)
            
            print(test_summary)

In [8]:
Epoch = 300
for epoch in range(1, Epoch + 1):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 0


OutOfMemoryError: CUDA out of memory. Tried to allocate 26.00 MiB (GPU 0; 6.00 GiB total capacity; 5.22 GiB already allocated; 0 bytes free; 5.36 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF